# P0 · 지도·그래프 공간 통합 탐색
기존 실험 ①~④의 완료 결과로 행정동→분석 격자→관측 위치를 탐색합니다.

이 노트북은 기존 준비·실험을 실행하지 않습니다. `docs/과업3_P0_공간탐색환경.md`에 실행 방법·공간 단위·공통 위치 기준·한계가 있습니다.

좌표계는 공급자 미확인 EPSG:5179 임시 가정입니다. 분석 격자는 통신사 격자/주민 생활권을 의미하지 않으며 유동 값 합계는 고유 방문자 수가 아닙니다.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = None  # 다른 PC이면 프로젝트 루트만 지정
if PROJECT_ROOT is None:
    PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Code' / 'sk_admin_mapping' / 'spatial_explorer.py').is_file())
PROJECT_ROOT = Path(PROJECT_ROOT).resolve()
WORK = PROJECT_ROOT / 'Code' / 'sk_admin_mapping'
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
from spatial_explorer import SpatialExplorer
explorer = SpatialExplorer()  # 완료된 새 공간 캐시만 확인·읽기
print('세 자료 공통 유효 위치:', f"{explorer.meta['n_joint']:,}")
print('추천 격자:', {explorer.meta['city_names'][k]: f'{v}m' for k,v in explorer.meta['recommended'].items()})

## 탐색 화면 실행
아래 셀은 로컬 서버가 이미 실행 중이면 재사용합니다. 새 서버는 화면 없이 실행하고 로그를 `outputs/validation/p0_runtime`에 저장합니다. 지도 라이브러리·배경지도는 인터넷 연결이 필요합니다.

공간을 클릭해 상세 그래프를 확인한 뒤 **선택 공간의 하위 단위 탐색**을 누릅니다. 같은 공간에서 실험·지표·월·위치 기준을 바꿔 비교할 수 있습니다.

In [ ]:
import subprocess
import urllib.request
import json
import time
import os
from IPython.display import display, HTML
PORT = 8766
URL = f'http://127.0.0.1:{PORT}'
def p0_alive():
    try:
        with urllib.request.urlopen(URL + '/api/catalog', timeout=2) as r:
            return 'n_joint' in json.load(r)
    except (OSError, ValueError):
        return False
p0_server = None
if not p0_alive():
    logdir = WORK / 'outputs' / 'validation' / 'p0_runtime'
    logdir.mkdir(parents=True, exist_ok=True)
    with (logdir / 'server.stdout.log').open('w',encoding='utf-8') as out, (logdir / 'server.stderr.log').open('w',encoding='utf-8') as err:
        p0_server = subprocess.Popen([sys.executable, '-B', '-u', str(WORK / 'spatial_explorer.py'), '--serve', '--port', str(PORT)], cwd=PROJECT_ROOT,
                                     stdout=out,stderr=err,creationflags=subprocess.CREATE_NO_WINDOW if os.name=='nt' else 0)
    for _ in range(120):
        if p0_alive():
            break
        if p0_server.poll() is not None:
            raise RuntimeError('서버 시작 실패. p0_runtime/server.stderr.log를 확인하세요.')
        time.sleep(.25)
    else:
        raise RuntimeError('서버 응답 대기 시간 초과. 로그를 확인하세요.')
display(HTML(f'<a href="{URL}" target="_blank">P0 지도·그래프 탐색 화면 열기</a>'))

## 종료 및 다음 단계
이 커널에서 새로 시작한 서버만 종료하려면 아래 선택 셀에서 `STOP_SERVER=True`로 바꿉니다. 다른 터미널에서 실행한 서버는 해당 터미널의 Ctrl+C로 종료합니다.

P1은 이 화면과 `outputs/canonical/spatial_explorer/coverage.csv`를 재사용해 실제 사례·해석 보고서를 작성합니다. P2의 외부 데이터 결합 검토는 별도 단계입니다.

In [ ]:
STOP_SERVER = False
if STOP_SERVER and p0_server is not None:
    p0_server.terminate()
    p0_server.wait(timeout=10)
    print('이 커널에서 시작한 P0 서버를 종료했습니다.')